# Toronto Building Permits — Active Permits Analysis
City of Toronto Open Data — 229 692 active building permits, issued from 1979 to present.
Covers all permit types: residential, commercial, mechanical, plumbing, demolition, and more.

In [1]:
from pathlib import Path

import altair as alt
import polars as pl

## 1. Load & Schema

In [2]:
RAW = Path("../data/raw/building-permits-active-permits/building-permits-active-permits.csv")

df = pl.read_csv(
    RAW,
    null_values=[""],
    infer_schema_length=10_000,
    schema_overrides={
        "GEO_ID": pl.Utf8,
        "STREET_NUM": pl.Utf8,
        "ASSEMBLY": pl.Float64,
        "INSTITUTIONAL": pl.Float64,
        "RESIDENTIAL": pl.Float64,
        "BUSINESS_AND_PERSONAL_SERVICES": pl.Float64,
        "MERCANTILE": pl.Float64,
        "INDUSTRIAL": pl.Float64,
        "INTERIOR_ALTERATIONS": pl.Float64,
        "DEMOLITION": pl.Float64,
        "DWELLING_UNITS_CREATED": pl.Utf8,
        "DWELLING_UNITS_LOST": pl.Utf8,
    },
)

df = df.with_columns(
    pl.col("APPLICATION_DATE").str.to_date(format="%Y-%m-%d", strict=False),
    pl.col("ISSUED_DATE").str.to_date(format="%Y-%m-%d", strict=False),
    pl.col("DWELLING_UNITS_CREATED").str.strip_chars().cast(pl.Int32, strict=False),
    pl.col("DWELLING_UNITS_LOST").str.strip_chars().cast(pl.Int32, strict=False),
).with_columns(
    pl.col("APPLICATION_DATE").dt.year().alias("app_year"),
    pl.col("APPLICATION_DATE").dt.month().cast(pl.Int32).alias("app_month"),
)

print(f"Rows: {len(df):,}   Columns: {df.width}")
df.schema

Rows: 229,692   Columns: 34


Schema([('_id', Int64),
        ('PERMIT_NUM', String),
        ('REVISION_NUM', Int64),
        ('PERMIT_TYPE', String),
        ('STRUCTURE_TYPE', String),
        ('WORK', String),
        ('STREET_NUM', String),
        ('STREET_NAME', String),
        ('STREET_TYPE', String),
        ('STREET_DIRECTION', String),
        ('POSTAL', String),
        ('GEO_ID', String),
        ('WARD_GRID', String),
        ('APPLICATION_DATE', Date),
        ('ISSUED_DATE', Date),
        ('COMPLETED_DATE', String),
        ('STATUS', String),
        ('DESCRIPTION', String),
        ('CURRENT_USE', String),
        ('PROPOSED_USE', String),
        ('DWELLING_UNITS_CREATED', Int32),
        ('DWELLING_UNITS_LOST', Int32),
        ('EST_CONST_COST', String),
        ('ASSEMBLY', Float64),
        ('INSTITUTIONAL', Float64),
        ('RESIDENTIAL', Float64),
        ('BUSINESS_AND_PERSONAL_SERVICES', Float64),
        ('MERCANTILE', Float64),
        ('INDUSTRIAL', Float64),
        ('INTERIOR_ALTE

In [3]:
df.head(5)

_id,PERMIT_NUM,REVISION_NUM,PERMIT_TYPE,STRUCTURE_TYPE,WORK,STREET_NUM,STREET_NAME,STREET_TYPE,STREET_DIRECTION,POSTAL,GEO_ID,WARD_GRID,APPLICATION_DATE,ISSUED_DATE,COMPLETED_DATE,STATUS,DESCRIPTION,CURRENT_USE,PROPOSED_USE,DWELLING_UNITS_CREATED,DWELLING_UNITS_LOST,EST_CONST_COST,ASSEMBLY,INSTITUTIONAL,RESIDENTIAL,BUSINESS_AND_PERSONAL_SERVICES,MERCANTILE,INDUSTRIAL,INTERIOR_ALTERATIONS,DEMOLITION,BUILDER_NAME,app_year,app_month
i64,str,i64,str,str,str,str,str,str,str,str,str,str,date,date,str,str,str,str,str,i32,i32,str,f64,f64,f64,f64,f64,f64,f64,f64,str,i32,i32
1,"""00 326851 CMB""",0,"""Residential Building Permit""","""SFD - Townhouse""","""New Building""","""64""","""FINCH""","""AVE""","""W""",""" """,null,"""N1822""",2000-08-10,2000-12-05,null,"""Inspection""","""NEW TOWNHOUSE""",null,null,0,0,"""200,000""",0.0,0.0,192.45,0.0,0.0,0.0,0.0,0.0,null,2000,8
2,"""16 132796 BLD""",0,"""Building Additions/Alterations""","""Multiple Unit Building""","""Other(BA)""","""1267""","""QUEEN""","""ST""","""W""","""M6K""","""10775305""","""S0436""",2016-03-29,2016-05-26,null,"""Permit Issued""","""Proposal to remove roof and wa…","""Mixed-Use""","""Mixed-Use""",0,0,"""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,2016,3
3,"""08 225597 PLB""",0,"""Plumbing(PS)""","""SFD - Detached""","""Building Permit Related(PS)""","""22""","""HUMEWOOD""","""DR""",""" ""","""M6C""","""9278430""","""S1229""",2008-11-28,2008-12-03,null,"""Inspection""","""Plumbing - Interior alteration…","""Sfd""","""Sfd""",null,null,"""DO NOT UPDATE OR DELETE THIS I…",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,2008,11
4,"""04 118465 BLD""",0,"""Small Residential Projects""","""SFD - Semi-Detached""","""Multiple Projects""","""16""","""ELVINA""","""GDNS""",""" ""","""M4P""","""7635758""","""N1526""",2004-03-23,2004-03-23,null,"""Permit Issued""","""Underpinning, new basement flo…",null,null,0,0,"""18000""",0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,null,2004,3
5,"""25 164063 HVA""",1,"""Mechanical(MS)""","""Office""","""Building Permit Related(MS)""","""111""","""STEINWAY""","""BLVD""",""" ""","""M9W""","""30097075""","""W0121""",2025-10-27,2025-11-18,null,"""Revision Issued""","""REV HVAC - PROPOSED OFFICE + M…","""Gray Shell""","""Office """,null,null,"""DO NOT UPDATE OR DELETE THIS I…",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,2025,10


In [4]:
summary = pl.DataFrame({
    "column": df.columns,
    "dtype": [str(d) for d in df.dtypes],
    "non_null": [df[c].drop_nulls().len() for c in df.columns],
    "null_pct": [round(df[c].null_count() / len(df) * 100, 1) for c in df.columns],
})
summary

column,dtype,non_null,null_pct
str,str,i64,f64
"""_id""","""Int64""",229692,0.0
"""PERMIT_NUM""","""String""",229692,0.0
"""REVISION_NUM""","""Int64""",229692,0.0
"""PERMIT_TYPE""","""String""",229692,0.0
"""STRUCTURE_TYPE""","""String""",228827,0.4
…,…,…,…
"""INTERIOR_ALTERATIONS""","""Float64""",229692,0.0
"""DEMOLITION""","""Float64""",229692,0.0
"""BUILDER_NAME""","""String""",11642,94.9


## 2. Permit Applications Over Time

Year-over-year count of permit applications from 2000 onwards. 2026 is excluded as a partial year (data through May 2026 only).

In [5]:
by_year = (
    df.filter(
        (pl.col("app_year") >= 2000) & (pl.col("app_year") <= 2025)
    )
    .group_by("app_year")
    .agg(pl.len().alias("applications"))
    .sort("app_year")
)

alt.Chart(by_year).mark_bar(color="#4C72B0").encode(
    x=alt.X("app_year:O", title="Year"),
    y=alt.Y("applications:Q", title="Permit Applications"),
    tooltip=["app_year", "applications"],
).properties(title="Permit Applications Per Year (2000\u20132025)", width=720, height=280)

alt.Chart(...)

## 3. What Types of Permits Are Filed?

Plumbing and Small Residential Projects dominate the dataset, each accounting for roughly 22 % of all active permits.

In [6]:
permit_types = (
    df.group_by("PERMIT_TYPE")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
    .head(15)
)

alt.Chart(permit_types).mark_bar(color="#4C72B0").encode(
    x=alt.X("count:Q", title="Number of Permits"),
    y=alt.Y("PERMIT_TYPE:N", sort="-x", title=None),
    tooltip=["PERMIT_TYPE", "count"],
).properties(title="Top 15 Permit Types", width=660, height=340)

alt.Chart(...)

## 4. What Work Is Being Done?

Interior Alterations and New Building are the dominant construction activities, followed by mechanical and plumbing work tied to other permit filings.

In [7]:
work_types = (
    df.filter(pl.col("WORK").is_not_null())
    .group_by("WORK")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
    .head(15)
)

alt.Chart(work_types).mark_bar(color="#55A868").encode(
    x=alt.X("count:Q", title="Number of Permits"),
    y=alt.Y("WORK:N", sort="-x", title=None),
    tooltip=["WORK", "count"],
).properties(title="Top 15 Work Types", width=660, height=340)

alt.Chart(...)

## 5. Building Structure Types

Single-family detached homes are the most permitted structure type by far, reflecting Toronto's large stock of low-rise residential properties.

In [8]:
struct_types = (
    df.filter(pl.col("STRUCTURE_TYPE").is_not_null())
    .group_by("STRUCTURE_TYPE")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
    .head(15)
)

alt.Chart(struct_types).mark_bar(color="#DD8452").encode(
    x=alt.X("count:Q", title="Number of Permits"),
    y=alt.Y("STRUCTURE_TYPE:N", sort="-x", title=None),
    tooltip=["STRUCTURE_TYPE", "count"],
).properties(title="Top 15 Building Structure Types", width=660, height=340)

alt.Chart(...)

## 6. How Long Does It Take to Get a Permit?

Time from application submission to permit issuance. The median wait is about 30 days, but complex projects can take over a year.

In [9]:
df_times = (
    df.filter(
        pl.col("APPLICATION_DATE").is_not_null() &
        pl.col("ISSUED_DATE").is_not_null()
    )
    .with_columns(
        (pl.col("ISSUED_DATE") - pl.col("APPLICATION_DATE")).dt.total_days().alias("days_to_issue")
    )
    .filter(
        (pl.col("days_to_issue") >= 0) & (pl.col("days_to_issue") <= 730)
    )
)

print(f"Permits with both dates: {len(df_times):,}")
print(f"Median: {df_times['days_to_issue'].median():.0f} days")
print(f"P75:    {df_times['days_to_issue'].quantile(0.75):.0f} days")
print(f"P90:    {df_times['days_to_issue'].quantile(0.90):.0f} days")

band_order = ["\u2264 1 week", "1\u20134 weeks", "1\u20133 months",
               "3\u20136 months", "6\u201312 months", "> 1 year"]

buckets = (
    df_times.with_columns(
        pl.when(pl.col("days_to_issue") <= 7).then(pl.lit("\u2264 1 week"))
          .when(pl.col("days_to_issue") <= 30).then(pl.lit("1\u20134 weeks"))
          .when(pl.col("days_to_issue") <= 90).then(pl.lit("1\u20133 months"))
          .when(pl.col("days_to_issue") <= 180).then(pl.lit("3\u20136 months"))
          .when(pl.col("days_to_issue") <= 365).then(pl.lit("6\u201312 months"))
          .otherwise(pl.lit("> 1 year"))
          .alias("wait_band")
    )
    .group_by("wait_band")
    .agg(pl.len().alias("count"))
)

alt.Chart(buckets).mark_bar(color="#8172B3").encode(
    x=alt.X("wait_band:N", sort=band_order, title="Wait Time"),
    y=alt.Y("count:Q", title="Number of Permits"),
    tooltip=["wait_band", "count"],
).properties(title="Processing Time: Application to Permit Issuance", width=620, height=280)

Permits with both dates: 209,129
Median: 30 days
P75:    69 days
P90:    178 days


alt.Chart(...)

In [10]:
median_by_type = (
    df_times.group_by("PERMIT_TYPE")
    .agg(
        pl.col("days_to_issue").median().alias("median_days"),
        pl.len().alias("count"),
    )
    .filter(pl.col("count") >= 100)
    .sort("median_days", descending=True)
)

alt.Chart(median_by_type).mark_bar(color="#8172B3").encode(
    x=alt.X("median_days:Q", title="Median Days to Issuance"),
    y=alt.Y("PERMIT_TYPE:N", sort="-x", title=None),
    tooltip=["PERMIT_TYPE", "median_days", "count"],
).properties(
    title="Median Processing Time by Permit Type (\u2265 100 permits)",
    width=660, height=280
)

alt.Chart(...)

## 7. New Housing Units Created

The DWELLING_UNITS_CREATED field counts net new residential units approved per permit. This tracks Toronto's housing pipeline — laneway suites, condo towers, and everything in between.

In [11]:
housing = (
    df.filter(
        (pl.col("DWELLING_UNITS_CREATED") > 0) &
        (pl.col("app_year") >= 2010) &
        (pl.col("app_year") <= 2025)
    )
    .group_by("app_year")
    .agg(
        pl.col("DWELLING_UNITS_CREATED").sum().alias("units_created"),
        pl.len().alias("permits"),
    )
    .sort("app_year")
)

c1 = alt.Chart(housing).mark_bar(color="#55A868").encode(
    x=alt.X("app_year:O", title="Year"),
    y=alt.Y("units_created:Q", title="Dwelling Units Created"),
    tooltip=["app_year", "units_created", "permits"],
).properties(title="New Dwelling Units Approved per Year", width=660, height=220)

c2 = alt.Chart(housing).mark_bar(color="#4C72B0").encode(
    x=alt.X("app_year:O", title="Year"),
    y=alt.Y("permits:Q", title="Permits"),
    tooltip=["app_year", "permits"],
).properties(title="Number of Permits Creating New Units", width=660, height=180)

alt.vconcat(c1, c2).properties(spacing=10)

alt.VConcatChart(...)

## 8. Monthly Seasonality of Applications

Spring and early summer are peak permitting months, as homeowners and contractors plan renovations before the construction season.

In [12]:
MONTH_ABBR = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

by_month = (
    df.filter((pl.col("app_year") >= 2010) & (pl.col("app_year") <= 2025))
    .with_columns(
        pl.col("app_month").map_elements(
            lambda m: MONTH_ABBR[m - 1], return_dtype=pl.Utf8
        ).alias("month_name")
    )
    .group_by(["app_month", "month_name"])
    .agg(pl.len().alias("applications"))
    .sort("app_month")
)

alt.Chart(by_month).mark_bar(color="#4C72B0").encode(
    x=alt.X("month_name:N", sort=MONTH_ABBR, title="Month"),
    y=alt.Y("applications:Q", title="Applications (2010\u20132025)"),
    tooltip=["month_name", "applications"],
).properties(
    title="Permit Applications by Month \u2014 Seasonality (2010\u20132025)",
    width=660, height=280
)

alt.Chart(...)

## 9. COVID-19's Impact on Permit Activity

Monthly application counts from 2018 to 2024 reveal a sharp drop in spring 2020 when Toronto entered lockdown, followed by a strong rebound driven by home-renovation demand.

In [13]:
monthly_ts = (
    df.filter(
        (pl.col("app_year") >= 2018) & (pl.col("app_year") <= 2024)
    )
    .with_columns(
        (
            pl.col("app_year").cast(pl.Utf8) + "-" +
            pl.col("app_month").map_elements(lambda m: str(m).zfill(2), return_dtype=pl.Utf8)
        ).alias("year_month")
    )
    .group_by("year_month")
    .agg(pl.len().alias("applications"))
    .sort("year_month")
)

ym_order = monthly_ts["year_month"].to_list()

alt.Chart(monthly_ts).mark_line(color="#C44E52", point=True).encode(
    x=alt.X("year_month:N", sort=ym_order, title="Month",
            axis=alt.Axis(labelAngle=-60, labelOverlap=True, tickCount=24)),
    y=alt.Y("applications:Q", title="Applications"),
    tooltip=["year_month", "applications"],
).properties(
    title="Monthly Permit Applications 2018\u20132024 \u2014 COVID-19 Drop Visible in 2020",
    width=800, height=280
)

/var/folders/wq/skd882j579l1f8cqz1jgt5vm0000gn/T/ipykernel_58606/591670215.py:8: PolarsInefficientMapWarning: 
Expr.map_elements is significantly slower than the native expressions API.
Only use if you absolutely CANNOT implement your logic otherwise.
Replace this expression...
  - pl.col("app_month").map_elements(lambda m: ...)
with this one instead:
  + pl.col("app_month").cast(pl.String).str.zfill(2)

  pl.col("app_month").map_elements(lambda m: str(m).zfill(2), return_dtype=pl.Utf8)


alt.Chart(...)

## 10. Where Are the Most Permits? (Ward Grids)

Ward grid codes (e.g. N0823) map to city grid squares. North-central Toronto neighbourhoods consistently generate the highest permit volumes.

In [14]:
top_wards = (
    df.filter(pl.col("WARD_GRID").is_not_null())
    .group_by("WARD_GRID")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
    .head(20)
)

alt.Chart(top_wards).mark_bar(color="#DD8452").encode(
    x=alt.X("count:Q", title="Number of Permits"),
    y=alt.Y("WARD_GRID:N", sort="-x", title="Ward Grid"),
    tooltip=["WARD_GRID", "count"],
).properties(title="Top 20 Ward Grids by Permit Volume", width=660, height=420)

alt.Chart(...)

## 11. Construction Cost Distribution

45 % of permits have an invalid or missing cost estimate (either blank or the placeholder string \"DO NOT UPDATE\"). Among the valid 55 %, the majority are small residential jobs under \$50 000.

In [15]:
df_cost = (
    df.filter(
        pl.col("EST_CONST_COST").is_not_null() &
        ~pl.col("EST_CONST_COST").str.starts_with("DO NOT")
    )
    .with_columns(
        pl.col("EST_CONST_COST")
          .str.replace_all(",", "")
          .cast(pl.Float64, strict=False)
          .alias("cost_clean")
    )
    .filter(pl.col("cost_clean").is_not_null() & (pl.col("cost_clean") > 0))
)

print(f"Permits with valid non-zero cost: {len(df_cost):,} ({len(df_cost)/len(df):.1%} of total)")

cost_bands = [
    "< $10K", "$10K\u2013$50K", "$50K\u2013$100K",
    "$100K\u2013$500K", "$500K\u2013$1M", "$1M\u2013$10M", "> $10M",
]

cost_buckets = (
    df_cost.with_columns(
        pl.when(pl.col("cost_clean") < 10_000).then(pl.lit("< $10K"))
          .when(pl.col("cost_clean") < 50_000).then(pl.lit("$10K\u2013$50K"))
          .when(pl.col("cost_clean") < 100_000).then(pl.lit("$50K\u2013$100K"))
          .when(pl.col("cost_clean") < 500_000).then(pl.lit("$100K\u2013$500K"))
          .when(pl.col("cost_clean") < 1_000_000).then(pl.lit("$500K\u2013$1M"))
          .when(pl.col("cost_clean") < 10_000_000).then(pl.lit("$1M\u2013$10M"))
          .otherwise(pl.lit("> $10M"))
          .alias("cost_range")
    )
    .group_by("cost_range")
    .agg(pl.len().alias("count"))
)

alt.Chart(cost_buckets).mark_bar(color="#8172B3").encode(
    x=alt.X("cost_range:N", sort=cost_bands, title="Estimated Construction Cost"),
    y=alt.Y("count:Q", title="Number of Permits"),
    tooltip=["cost_range", "count"],
).properties(
    title="Construction Cost Distribution (valid non-zero values only)",
    width=660, height=280
)

Permits with valid non-zero cost: 105,013 (45.7% of total)


alt.Chart(...)

## 12. Permit Type Trends Over Time (Heatmap)

How the mix of permit types has shifted year by year since 2010. Darker squares indicate more permits of that type in that year.

## 13. New House Permits Per Year

Permits with `PERMIT_TYPE = 'New Houses'` track detached, semi-detached, and townhouse construction.
The series starts reliably from 2004; earlier years used different permit type names.
2026 is excluded as a partial year.

In [16]:
new_houses = (
    df.filter(
        (pl.col("PERMIT_TYPE") == "New Houses") &
        (pl.col("app_year") >= 2004) &
        (pl.col("app_year") <= 2025)
    )
    .group_by("app_year")
    .agg(pl.len().alias("permits"))
    .sort("app_year")
)

alt.Chart(new_houses).mark_bar(color="#55A868").encode(
    x=alt.X("app_year:O", title="Year"),
    y=alt.Y("permits:Q", title="New House Permits"),
    tooltip=["app_year", "permits"],
).properties(title="New House Permits Per Year (2004–2025)", width=720, height=300)

alt.Chart(...)

## 14. Renovation vs. New Build Split Over Time

Trades permits (plumbing, mechanical, drainage, fire/security) are excluded — they are filed
separately for every project and would dwarf the signal. The remaining permits are classified
by the WORK column: anything starting with "New Building" is a new build; "Demolition" is
tracked separately; everything else is renovation or addition to an existing structure.

In [18]:
TRADES_TYPES = [
    "Plumbing(PS)", "Mechanical(MS)", "Drain and Site Service",
    "Fire/Security Upgrade", "DCs DeferredFees",
]

cat_order = ["New Build", "Renovation / Addition", "Demolition"]
cat_colours = ["#55A868", "#4C72B0", "#C44E52"]

split_df = (
    df.filter(
        (pl.col("app_year") >= 2004) &
        (pl.col("app_year") <= 2025) &
        ~pl.col("PERMIT_TYPE").is_in(TRADES_TYPES)
    )
    .with_columns(
        pl.when(pl.col("WORK").str.starts_with("New Building"))
          .then(pl.lit("New Build"))
          .when(pl.col("WORK") == "Demolition")
          .then(pl.lit("Demolition"))
          .otherwise(pl.lit("Renovation / Addition"))
          .alias("category")
    )
    .group_by(["app_year", "category"])
    .agg(pl.len().alias("count"))
    .sort(["app_year", "category"])
)

base = alt.Chart(split_df).encode(
    x=alt.X("app_year:O", title="Year"),
    color=alt.Color(
        "category:N",
        scale=alt.Scale(domain=cat_order, range=cat_colours),
        legend=alt.Legend(title=None),
    ),
    tooltip=["app_year", "category", "count"],
)

c1 = base.mark_bar().encode(
    y=alt.Y("count:Q", title="Permits"),
).properties(
    title="New Build vs. Renovation — Absolute Permit Counts (2004–2025)",
    width=720, height=240,
)

c2 = base.mark_bar().encode(
    y=alt.Y("count:Q", stack="normalize", title="Share",
            axis=alt.Axis(format="%")),
).properties(
    title="Share of Permits by Category",
    width=720, height=180,
)

alt.vconcat(c1, c2).properties(spacing=10)

alt.VConcatChart(...)

In [17]:
top_8_types = (
    df.group_by("PERMIT_TYPE")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
    .head(8)
    ["PERMIT_TYPE"].to_list()
)

heat_df = (
    df.filter(
        pl.col("PERMIT_TYPE").is_in(top_8_types) &
        (pl.col("app_year") >= 2010) &
        (pl.col("app_year") <= 2025)
    )
    .group_by(["PERMIT_TYPE", "app_year"])
    .agg(pl.len().alias("count"))
    .sort(["app_year", "PERMIT_TYPE"])
    .with_columns(pl.col("app_year").cast(pl.Utf8))
)

year_order = [str(y) for y in range(2010, 2026)]

alt.Chart(heat_df).mark_rect().encode(
    x=alt.X("app_year:O", sort=year_order, title="Year"),
    y=alt.Y("PERMIT_TYPE:N", title=None),
    color=alt.Color("count:Q", scale=alt.Scale(scheme="blues"), title="Permits"),
    tooltip=["PERMIT_TYPE", "app_year", "count"],
).properties(
    title="Permit Volume by Type and Year (2010\u20132025)",
    width=720, height=300
)

alt.Chart(...)